# **Gikomba Market Stall Permofance Analyis**

**Prepared for:** Wanjiku, fresh-produce stall at Gikomba Market

**Analyst:** Amanda Ireri

Wanjiku has two weeks of daily sales and a rough stock record. This notebook uses core Python only to answer : how is the business doing, is it growing, what should be restocked, and where should she focus?

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700] #14 days, KES

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2


#qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg

stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions": {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma": {"qty": 9, "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost":50, "price": 75},
    "Avocado": {"qty": 12, "reorder": 10, "cost": 80, "price": 140}
}

##**Task 1: Summary statistics and best/worst days**

**Approach:** Use `sum()` , `max()` and `min()` for the totals. Then use `list.index` to find the *position* of the best and worst days. Positions 0-6 are week 1 and 7-13 are week 2, so `position // 7 + 1 ` gives the week number. The same position looks up the day name in the parallel `days` list

In [2]:
total_sales = sum(daily_sales)
average_sales = total_sales / len(daily_sales)
highest_sales = max(daily_sales)
lowest_sales = min(daily_sales)

best_pos = daily_sales.index(highest_sales)
worst_pos = daily_sales.index(lowest_sales)

best_day, best_week = days[best_pos], best_pos // 7 + 1
worst_day, worst_week = days[worst_pos], worst_pos // 7 + 1

print(f"Total sales : KES {total_sales:,}")
print(f"Average daily: KES {average_sales:,.2f}")
print(f"Highest_day: KES {highest_sales:,} ({best_day}, week {best_week})")
print(f"Lowest day: KES {lowest_sales:,} ({worst_day}, week{worst_week})")

Total sales : KES 46,450
Average daily: KES 3,317.86
Highest_day: KES 5,600 (Sat, week 2)
Lowest day: KES 1,950 (Tue, week1)


##**Task 2: Week totals and week-on-week growth**

**Approach:** `week_totals()` slices the list into week 1 (`[:7]`) and week 2 (`[7:]`) and returns both totals as a tuple. Growth is `(week 2 - week 1)` ÷ week 1 * 100

In [3]:
def week_totals(sales):
  """ Return (week 1 total, week 2 total) for a list of 14 daily sales."""
  week1 = sum(sales[:7])
  week2 = sum(sales[7:])
  return week1, week2

week1_total, week2_total = week_totals(daily_sales)
growth_pct = (week2_total - week1_total) / week1_total * 100

print (f"Week1 total: KES {week1_total:,}")
print (f"Week2 total : KES {week2_total:,}")
print (f"Week-on-week growth : {growth_pct:.1f}%")

Week1 total: KES 22,050
Week2 total : KES 24,400
Week-on-week growth : 10.7%


## **Task 3: Classifying each day as Strong, Normal or Weak**

**Approach** `classify_day()` compares a day's sales with the 14-day average. More than 20% above average is "Strong", more than 20% below average is "Weak", and anything else is "Normal". I apply it to everday and count the labels with a dictionary, using `.get(label, 0) + 1`.

In [4]:
THRESHOLD = 0.20

def classify_day (amount, average):

  """Classify a day's sales against the average."""

  if amount > average * (1 + THRESHOLD):
    return "Strong"
  elif amount < average * (1 - THRESHOLD):
    return "Weak"
  else:
    return "Normal"

day_counts = {}
for day, amount in zip(days, daily_sales):
  label = classify_day(amount, average_sales)
  day_counts[label] = day_counts.get(label, 0) + 1

for label in ["Strong", "Normal", "Weak"]:
  print(f"{label} : {day_counts.get(label, 0)} days")

Strong : 4 days
Normal : 4 days
Weak : 6 days


## **Task 4: Weekday vs weekend sales**

**Approach:** I loop through the days and put each amount into a weekday list (Mon - Fri) or a weekend list (Sat_Sun). Each group is averaged by its own number, then I compute the weekend_difference as a percentage of the weekday average

In [5]:
weekday_sales = []
weekend_sales = []

for day, amount in zip(days, daily_sales):
  if day in ("Sat", "Sun"):
    weekend_sales.append(amount)
  else:
    weekday_sales.append(amount)

weekday_avg = sum(weekday_sales) / len (weekday_sales)
weekend_avg = sum(weekend_sales) / len (weekend_sales)
weekend_uplift_pct = (weekend_avg - weekday_avg)/ weekday_avg * 100

print(f"Weekday average (Mon - Fri): KES {weekday_avg:,.2f}")
print (f"Weekend average (Sat-Sun): KES ({weekend_avg:,.2f})")
print (f"Weekends sell {weekend_uplift_pct:.1f}% more than weekdays")

Weekday average (Mon - Fri): KES 2,685.00
Weekend average (Sat-Sun): KES (4,900.00)
Weekends sell 82.5% more than weekdays


## **Task 5: Reorder list and gross margins**

**Approach:** `get_reorder_list()` loops through the stock dictionary with `.items()` and collects products where the quantity is at or below the reorder level. A second loop calculates each product's gross margin, `(price − cost) ÷ price × 100`, and stores it in a dictionary.

In [6]:
def get_reorder_list(stock_dict):
  """Return a list of product names whose quantity is at or below reorder level"""
  to_reorder = []
  for product, info in stock_dict.items():
    if info["qty"] <= info["reorder"]:
      to_reorder.append(product)
  return to_reorder

reorder_list = get_reorder_list(stock)
print("Restock now:", ", ".join(reorder_list))

margins = {}
for product, info in stock.items():
  margins[product] = (info["price"] - info["cost"]) / info["price"] * 100

for product, margin in margins.items():
  print(f"{product:<10} {margin:.1f}%")

Restock now: Tomatoes, Sukuma
Tomatoes   33.3%
Onions     30.0%
Sukuma     40.0%
Potatoes   33.3%
Avocado    42.9%


## **Task 6: Stall Performance Report**

**Approach:** Everything is printed from the variables computed above, using f-strings. Money uses `KES` with thousands separators (`:,`). Alignment specifiers (`<` left, `>` right, `^` centre) keep the columns tidy.

In [9]:
width = 46
print("=" * width)
print(f"{'STALL PERFORMANCE REPORT':^{width}}")
print("=" * width)

print("\nSALES OVERVIEW")
print(f"{'Total sales (14 days)':<26}KES {total_sales:>10,}")
print(f"{'Average per day':<26}KES {average_sales:>10,.2f}")
print(f"{'Best day':<26}KES {highest_sales:>10,}  ({best_day}, wk {best_week})")
print(f"{'Worst day':<26}KES {lowest_sales:>10,}  ({worst_day}, wk {worst_week})")

print("\nGROWTH")
print(f"{'Week 1 total':<26}KES {week1_total:>10,}")
print(f"{'Week 2 total':<26}KES {week2_total:>10,}")
print(f"{'Week-on-week growth':<26}{growth_pct:>+14.1f}%")

print("\nDAY PERFORMANCE")
for label in ["Strong", "Normal", "Weak"]:
    print(f"{label:<26}{day_counts.get(label, 0):>14} days")

print("\nWEEKDAY vs WEEKEND")
print(f"{'Weekday average':<26}KES {weekday_avg:>10,.2f}")
print(f"{'Weekend average':<26}KES {weekend_avg:>10,.2f}")
print(f"{'Weekend uplift':<26}{weekend_uplift_pct:>+14.1f}%")

print("\nSTOCK STATUS")
print(f"{'Product':<12}{'Qty':>5}{'Reorder':>9}{'Margin':>9}  Status")
for product, info in stock.items():
    status = "RESTOCK" if product in reorder_list else "OK"
    print(f"{product:<12}{info['qty']:>5}{info['reorder']:>9}{margins[product]:>8.1f}%  {status}")
print("=" * width)

           STALL PERFORMANCE REPORT           

SALES OVERVIEW
Total sales (14 days)     KES     46,450
Average per day           KES   3,317.86
Best day                  KES      5,600  (Sat, wk 2)
Worst day                 KES      1,950  (Tue, wk 1)

GROWTH
Week 1 total              KES     22,050
Week 2 total              KES     24,400
Week-on-week growth                +10.7%

DAY PERFORMANCE
Strong                                 4 days
Normal                                 4 days
Weak                                   6 days

WEEKDAY vs WEEKEND
Weekday average           KES   2,685.00
Weekend average           KES   4,900.00
Weekend uplift                     +82.5%

STOCK STATUS
Product       Qty  Reorder   Margin  Status
Tomatoes       18       25    33.3%  RESTOCK
Onions         40       20    30.0%  OK
Sukuma          9       15    40.0%  RESTOCK
Potatoes       55       30    33.3%  OK
Avocado        12       10    42.9%  OK


## **Recommendations for Wanjiku**

1. **Plan around weekends.** Weekends average KES 4,900 a day against KES 2,685 on weekdays, which is 82.5% more. Make sure stock is full before Saturday, and keep Saturday's best result (KES 5,600 in week 2) as the target.
2. **Restock Tomatoes and Sukuma now.** Tomatoes are at 18 kg against a reorder level of 25 kg, and Sukuma is at 9 kg against 15 kg. Sukuma earns a 40.0% margin, so running out costs her good profit.
3. **Push Avocado.** It has the highest margin at 42.9%, and 12 kg is only 2 kg above its reorder level of 10 kg. Keep it well stocked and displayed.
4. **Don't over-invest in Onions.** They have the lowest margin (30.0%) and 40 kg in stock, double the reorder level of 20 kg. Hold off buying more until stock falls.
5. **Build on the growth and fix weak weekdays.** Sales grew 10.7% from week 1 (KES 22,050) to week 2 (KES 24,400), but 6 of 14 days were "Weak". Most of those are weekdays, so a weekday offer would help, especially on Tuesday, the weakest day at KES 1,950.